# KNN — PCA

PCA fit은 Training만 사용. 기존 latency screen은 128장×3회.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("KNN", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
knn=model; tr=knn._fit_X; ty=knn._y; kbase=base
print("Local output:",root)

In [ ]:
from sklearn.decomposition import PCA
t=time.perf_counter()
pc=PCA(n_components=512,svd_solver='covariance_eigh',whiten=False)
pc.fit(tr)
pca_fit_sec=time.perf_counter()-t
print('PCA fit',pca_fit_sec,pc.explained_variance_ratio_.sum())
joblib.dump(pc,root/'03_KNN/pca512_training_only.joblib',compress=3)
np.savez(root/'03_KNN/pca_variance.npz',ratios=pc.explained_variance_ratio_)

In [ ]:
# PCA full training projection; nested prefixes share exact fitted eigenspace.
t=time.perf_counter()
zt=pc.transform(tr);zv=pc.transform(Xv);zo=pc.transform(Xo)
pca_transform_sec=time.perf_counter()-t
print(zt.shape,zt.dtype,pca_transform_sec)

In [ ]:
pca_rows=[{'condition':'PCA00','dims':1108,'explained_var':1.0,'scope':'full_training_full_handmade',**stats(kbase,kbase)}]
pca_preds={'PCA00':kbase}
for dd in [512,256,128,64]:
    ztrain=np.ascontiguousarray(zt[:,:dd]);zzv=np.ascontiguousarray(zv[:,:dd])
    t=time.perf_counter()
    mod=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(ztrain,ty)
    ft=time.perf_counter()-t;t=time.perf_counter();pred=mod.predict(zzv);et=time.perf_counter()-t
    rec={'condition':f'PCA{dd}','dims':dd,'explained_var':float(pc.explained_variance_ratio_[:dd].sum()),'scope':'full_training_full_handmade','pca_fit_s_shared':pca_fit_sec,'pca_transform_512_s_shared':pca_transform_sec,'knn_fit_s':ft,'batch_eval_s':et,**stats(pred,kbase)}
    pca_rows.append(rec);pca_preds[f'PCA{dd}']=pred
    print(rec['condition'],rec['correct'],rec['macro_f1'],rec['explained_var'],et,flush=True)
    del ztrain,zzv,mod
    gc.collect()
pd.DataFrame(pca_rows).to_csv(root/'03_KNN/EXP-FT-KNN-004_handmade.csv',index=False)
pd.DataFrame({'true':y,**pca_preds}).to_csv(root/'03_KNN/EXP-FT-KNN-004_predictions.csv',index=False)

In [ ]:
# Confirm official development score for PCA alternatives with equal accuracy;
# guardrail is separate from strict accuracy gate.
for dd in [128,512]:
 ztrain=np.ascontiguousarray(zt[:,:dd])
 mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(ztrain,ty)
 t=time.perf_counter()
 pp=[]
 for lo in range(0,len(yo),2400):pp.append(mm.predict(np.ascontiguousarray(zo[lo:lo+2400,:dd])))
 po=np.concatenate(pp);duration=time.perf_counter()-t
 row=next(r for r in pca_rows if r['dims']==dd)
 row['official_correct']=int((po==yo).sum());row['official_accuracy']=float(np.mean(po==yo));row['official_guardrail_pass']=row['official_correct']>=18856-19
 np.save(root/f'03_KNN/PCA{dd}_official_pred.npy',po)
 print(dd,row['official_correct'],row['official_guardrail_pass'],duration,flush=True)
 del ztrain,mm
 gc.collect()
pd.DataFrame(pca_rows).to_csv(root/'03_KNN/EXP-FT-KNN-004_handmade.csv',index=False)

In [ ]:
# Timing screen on a fixed 128-image subset, three passes, full preprocessing, same affinity and environment for all.
# Not a replacement for final 1280x5 timing.
rngtime=np.random.default_rng(42)
tim_idx=np.sort(np.concatenate([rngtime.choice(np.flatnonzero(y==c0),8,replace=False) for c0 in range(16)]))
np.save(root/'03_KNN/latency_screen_indices.npy',tim_idx)
tim_rows=[]
# baseline
def latency_screen(m,dd):
 def one(i):
  f=fb.hybrid(raw[i:i+1])
  if dd!=1108:f=pc.transform(f)[:,:dd].copy()
  return int(m.predict(f)[0])
 for i in tim_idx[:16]:one(i)
 reps=[]
 ref=None
 for rep in range(3):
  t=time.perf_counter();pred=np.array([one(i) for i in tim_idx]);elapsed=time.perf_counter()-t
  reps.append(elapsed*1000/len(tim_idx))
  if ref is None:ref=pred
  else:assert np.array_equal(ref,pred)
 return reps,ref
for dd in [1108,128,512]:
 if dd==1108:mm=knn
 else:mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(np.ascontiguousarray(zt[:,:dd]),ty)
 times,pred=latency_screen(mm,dd)
 ref=pca_preds['PCA00' if dd==1108 else f'PCA{dd}'][tim_idx]
 rec={'dims':dd,'n':len(tim_idx),'repeats':3,'mean_ms_per_image':float(np.mean(times)),'individual_ms':times,'matches_batch':int(np.sum(pred==ref)),'timing_scope':'screen only, same machine; record concurrent workload status'}
 tim_rows.append(rec)
 print(rec,flush=True)
 if dd!=1108:del mm
pd.DataFrame(tim_rows).to_csv(root/'03_KNN/PCA_latency_screen.csv',index=False)